In [ ]:
import pandas as pd

In [ ]:
listings = pd.read_json("listings.json")
property_att = pd.read_json("property_att.json")
agents = pd.read_json("agents.json")
buyers = pd.read_json("buyers.json")

listings.to_csv("listings.csv", index = False)
property_att.to_csv("property_att.csv", index = False)
agents.to_csv("agents.csv", index = False)
buyers.to_csv("buyers.csv", index = False)

sales = pd.read_csv("sales.csv")

In [ ]:
import sqlite3

conn = sqlite3.connect("brickview_realestate.db")
cursor = conn.cursor()

In [ ]:
create_listings_table = """
CREATE TABLE IF NOT EXISTS listings(
    Listing_id TEXT PRIMARY KEY,
    City TEXT,
    Property_Type TEXT,
    Price REAL,
    Sqft REAL,
    Date_Listed TEXT,
    Agent_ID TEXT,
    Latitude REAL,
    Longitude REAL
);
"""

create_properties_table = """
CREATE TABLE IF NOT EXISTS property_att(
    attribute_id INTEGER PRIMARY KEY,
    listing_id TEXT,
    bedrooms INTEGER,
    bathrooms INTEGER,
    floor_number INTEGER,
    total_floors INTEGER,
    year_built INTEGER,
    is_rented TEXT,
    tenant_count INTEGER,
    furnishing_status TEXT,
    metro_distance_km REAL,
    parking_available TEXT,
    power_backup TEXT
);
"""

create_buyers_table = """
CREATE TABLE IF NOT EXISTS buyers(
    buyer_id TEXT PRIMARY KEY,
    sale_id TEXT,
    buyer_type TEXT,
    payment_mode TEXT,
    loan_taken TEXT,
    loan_provider TEXT,
    loan_amount REAL
);
"""

create_agents_table = """
CREATE TABLE IF NOT EXISTS agents(
    Agent_ID TEXT PRIMARY KEY,
    Name TEXT,
    Phone TEXT,
    Email TEXT,
    commission_rate REAL,
    deals_closed INTEGER,
    rating REAL,
    experience_years INTEGER,
    avg_closing_days INTEGER
);
"""

create_sales_table = """
CREATE TABLE IF NOT EXISTS sales(
    Listing_ID TEXT PRIMARY KEY,
    Sale_Price REAL,
    Date_Sold TEXT,
    Days_on_Market INTEGER
);
"""

cursor.execute(create_listings_table)
cursor.execute(create_properties_table)
cursor.execute(create_buyers_table)
cursor.execute(create_agents_table)
cursor.execute(create_sales_table)

conn.commit()

print("All tables created successfully")

In [ ]:
for _, row in listings.iterrows():
    cursor.execute
    ("""
        INSERT INTO listings(
            Listing_id, City, Property_Type, Price, Sqft,
            Date_Listed, Agent_ID, Latitude, Longitude
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["Listing_ID"],
        row["City"],
        row["Property_Type"],
        row["Price"],
        row["Sqft"],
        row["Date_Listed"],
        row["Agent_ID"],
        row["Latitude"],
        row["Longitude"]
    ))

conn.commit()

print("Listings inserted:", cursor.execute(
    "SELECT COUNT(*) FROM listings"
).fetchone()[0])

In [ ]:
for _, row in property_att.iterrows():
    cursor.execute
    ("""
        INSERT INTO property_att(
            attribute_id, listing_id, bedrooms, bathrooms, floor_number,
            total_floors, year_built, is_rented, tenant_count,
            furnishing_status, metro_distance_km,
            parking_available, power_backup
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["attribute_id"],
        row["listing_id"],
        row["bedrooms"],
        row["bathrooms"],
        row["floor_number"],
        row["total_floors"],
        row["year_built"],
        row["is_rented"],
        row["tenant_count"],
        row["furnishing_status"],
        row["metro_distance_km"],
        row["parking_available"],
        row["power_backup"]
    ))

conn.commit()

print("Property records:",
      cursor.execute("SELECT COUNT(*) FROM property_att").fetchone()[0])

In [ ]:
for _, row in agents.iterrows():
    cursor.execute
    ("""
        INSERT INTO agents(
            Agent_ID, Name, Phone, Email, commission_rate, deals_closed,
            rating, experience_years, avg_closing_days
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["Agent_ID"],
        row["Name"],
        row["Phone"],
        row["Email"],
        row["commission_rate"],
        row["deals_closed"],
        row["rating"],
        row["experience_years"],
        row["avg_closing_days"]
    ))

conn.commit()

print("Agents records:",
      cursor.execute("SELECT COUNT(*) FROM agents").fetchone()[0])

In [ ]:
for _, row in buyers.iterrows():
    cursor.execute
    ("""
        INSERT INTO buyers(
            buyer_id, sale_id, buyer_type, payment_mode,
            loan_taken, loan_provider, loan_amount
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        row["buyer_id"],
        row["sale_id"],
        row["buyer_type"],
        row["payment_mode"],
        row["loan_taken"],
        row["loan_provider"],
        row["loan_amount"]
    ))

conn.commit()

print("Buyers inserted:", cursor.execute(
    "SELECT COUNT(*) FROM buyers"
).fetchone()[0])

In [ ]:
for _, row in sales.iterrows():
    cursor.execute
    ("""
        INSERT INTO sales(
            Listing_ID, Sale_Price, Date_Sold, Days_on_Market
        )
        VALUES (?, ?, ?, ?)
    """, (
        row["Listing_ID"],
        row["Sale_Price"],
        row["Date_Sold"],
        row["Days_on_Market"]
    ))

conn.commit()

print("Sales inserted:", cursor.execute(
    "SELECT COUNT(*) FROM sales"
).fetchone()[0])

SQL Questions and Queries

Ques.1  What is the average listing price by city?

In [ ]:
query = """
SELECT City, AVG(Price) AS avg_listing_price
FROM listings
GROUP BY City;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.2 What is the average price per square foot by property type?

In [ ]:
query = """
SELECT Property_Type, AVG(Price / Sqft) AS Average_Price
FROM listings
GROUP BY Property_Type;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.3 How does furnishing status impact property prices?

In [ ]:
query = """
SELECT pa.furnishing_status, AVG(l.Price) AS Average_Property_Price
FROM property_att pa
JOIN listings l
    ON pa.listing_id = l.Listing_id
GROUP BY pa.furnishing_status;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.4 Do properties closer to metro stations command higher prices?

In [ ]:
query = """
SELECT CASE
        WHEN pa.metro_distance_km <= 2 THEN '0-2 km'
        WHEN pa.metro_distance_km <= 5 THEN '2-5 km'
        WHEN pa.metro_distance_km <= 10 THEN '5-10 km'
        ELSE '10+ km'
    END AS Metro_Distance_Bucket,
    AVG(l.Price) AS Average_Property_Price
FROM property_att pa
JOIN listings l
    ON pa.listing_id = l.Listing_id
GROUP BY Metro_Distance_Bucket
ORDER BY Average_Property_Price DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.5 Are rented properties priced differently from non-rented ones?

In [ ]:
query = """
SELECT
    pa.is_rented,
    AVG(l.Price) AS Average_Property_Price
FROM property_att pa
JOIN listings l
    ON pa.listing_id = l.Listing_id
GROUP BY pa.is_rented
ORDER BY Average_Property_Price DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.6 How do bedrooms and bathrooms affect pricing?

In [ ]:
query = """
SELECT
    pa.is_rented,
    AVG(l.Price) AS Average_Property_Price
FROM property_att pa
JOIN listings l
    ON pa.listing_id = l.Listing_id
GROUP BY pa.is_rented
ORDER BY Average_Property_Price DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.7 Do properties with parking and power backup sell at higher prices?

In [ ]:
query = """
SELECT
    pa.parking_available,
    pa.power_backup,
    AVG(s.Sale_Price) AS Average_Sale_Price
FROM property_att pa
JOIN sales s
    ON pa.listing_id = s.Listing_ID
GROUP BY pa.parking_available, pa.power_backup
ORDER BY Average_Sale_Price DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.8 How does year built influence listing price?

In [ ]:
query = """
SELECT
    pa.year_built,
    AVG(l.Price) AS Average_Listing_Price
FROM property_att pa
JOIN listings l
    ON pa.listing_id = l.Listing_id
GROUP BY pa.year_built
ORDER BY pa.year_built;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.9 Which cities have the highest average property prices?

In [ ]:
query = """
SELECT
    City,
    AVG(Price) AS Average_Property_Price
FROM listings
GROUP BY City
ORDER BY Average_Property_Price DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.10 How are properties distributed across price buckets?

In [ ]:
query = """
SELECT
    CASE
        WHEN Price < 500000 THEN 'Below 5 Lakh'
        WHEN Price < 1000000 THEN '5-10 Lakh'
        WHEN Price < 2500000 THEN '10-25 Lakh'
        WHEN Price < 5000000 THEN '25-50 Lakh'
        ELSE '50 Lakh+'
    END AS Price_Bucket,
    COUNT(*) AS Property_Count
FROM listings
GROUP BY Price_Bucket
ORDER BY Property_Count DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.11 What is the average days on market by city?

In [ ]:
query = """
SELECT
    l.City,
    AVG(s.Days_on_Market) AS Average_Days_on_Market
FROM listings l
JOIN sales s
    ON l.Listing_id = s.Listing_ID
GROUP BY l.City
ORDER BY Average_Days_on_Market;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.12 Which property types sell the fastest?

In [ ]:
query = """
SELECT
    l.Property_Type,
    AVG(s.Days_on_Market) AS Average_Days_on_Market
FROM listings l
JOIN sales s
    ON l.Listing_id = s.Listing_ID
GROUP BY l.Property_Type
ORDER BY Average_Days_on_Market ASC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.13 What percentage of properties are sold above listing price?

In [ ]:
query = """
SELECT
    ROUND(
        100.0 * SUM(
            CASE
                WHEN s.Sale_Price > l.Price THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS Percentage_Sold_Above_Listing_Price
FROM listings l
JOIN sales s
    ON l.Listing_id = s.Listing_ID;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.14 What is the sale-to-list price ratio by city?

In [ ]:
query = """
SELECT
    l.City,
    ROUND(AVG(s.Sale_Price * 1.0 / l.Price), 2) AS Sale_to_List_Ratio
FROM listings l
JOIN sales s
    ON l.Listing_id = s.Listing_ID
GROUP BY l.City
ORDER BY Sale_to_List_Ratio DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.15 Which listings took more than 90 days to sell?

In [ ]:
query = """
SELECT
    s.Listing_ID,
    l.City,
    l.Property_Type,
    l.Price AS Listing_Price,
    s.Sale_Price,
    s.Days_on_Market
FROM sales s
JOIN listings l
    ON s.Listing_ID = l.Listing_id
WHERE s.Days_on_Market > 90
ORDER BY s.Days_on_Market DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.16 How does metro distance affect time on market?

In [ ]:
query = """
SELECT
    CASE
        WHEN pa.metro_distance_km <= 2 THEN '0-2 km'
        WHEN pa.metro_distance_km <= 5 THEN '2-5 km'
        WHEN pa.metro_distance_km <= 10 THEN '5-10 km'
        ELSE '10+ km'
    END AS Metro_Distance_Bucket,
    AVG(s.Days_on_Market) AS Average_Days_on_Market
FROM property_att pa
JOIN sales s
    ON pa.listing_id = s.Listing_ID
GROUP BY Metro_Distance_Bucket
ORDER BY Average_Days_on_Market ASC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.17 What is the monthly sales trend?

In [ ]:
query = """
SELECT
    strftime('%Y-%m', Date_Sold) AS Sale_Month,
    COUNT(*) AS Number_of_Sales,
    SUM(Sale_Price) AS Total_Sales_Revenue
FROM sales
GROUP BY Sale_Month
ORDER BY Sale_Month;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.18 Which properties are currently unsold?

In [ ]:
query = """
SELECT
    l.Listing_id,
    l.City,
    l.Property_Type,
    l.Price AS Listing_Price,
    l.Sqft,
    l.Date_Listed
FROM listings l
LEFT JOIN sales s
    ON l.Listing_id = s.Listing_ID
WHERE s.Listing_ID IS NULL
ORDER BY l.Date_Listed;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.19 Which agents have closed the most sales?

In [ ]:
query = """
SELECT
    a.Agent_ID,
    a.Name,
    COUNT(s.Listing_ID) AS Total_Sales_Closed
FROM agents a
JOIN listings l
    ON a.Agent_ID = l.Agent_ID
JOIN sales s
    ON l.Listing_id = s.Listing_ID
GROUP BY a.Agent_ID, a.Name
ORDER BY Total_Sales_Closed DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.20 Who are the top agents by total sales revenue?

In [ ]:
query = """
SELECT
    a.Agent_ID,
    a.Name,
    SUM(s.Sale_Price) AS Total_Sales_Revenue
FROM agents a
JOIN listings l
    ON a.Agent_ID = l.Agent_ID
JOIN sales s
    ON l.Listing_id = s.Listing_ID
GROUP BY a.Agent_ID, a.Name
ORDER BY Total_Sales_Revenue DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.21 Which agents close deals fastest?

In [ ]:
query = """
SELECT
    Agent_ID,
    Name,
    avg_closing_days AS Average_Closing_Days
FROM agents
ORDER BY avg_closing_days ASC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.22 Does experience correlate with deals closed?

In [ ]:
query = """
SELECT
    experience_years,
    deals_closed
FROM agents
ORDER BY experience_years ASC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.23 Do agents with higher ratings close deals faster?

In [ ]:
query = """
SELECT
    Agent_ID,
    Name,
    rating,
    avg_closing_days AS Average_Closing_Days
FROM agents
ORDER BY rating DESC, Average_Closing_Days ASC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.24 What is the average commission earned by each agent?

In [ ]:
query = """
SELECT
    a.Agent_ID,
    a.Name,
    AVG(s.Sale_Price * a.commission_rate / 100.0) AS Average_Commission_Earned
FROM agents a
JOIN listings l
    ON a.Agent_ID = l.Agent_ID
JOIN sales s
    ON l.Listing_id = s.Listing_ID
GROUP BY a.Agent_ID, a.Name
ORDER BY Average_Commission_Earned DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.25 Which agents currently have the most active listings?

In [ ]:
query = """
SELECT
    a.Agent_ID,
    a.Name,
    COUNT(l.Listing_id) AS Active_Listings
FROM agents a
JOIN listings l
    ON a.Agent_ID = l.Agent_ID
LEFT JOIN sales s
    ON l.Listing_id = s.Listing_ID
WHERE s.Listing_ID IS NULL
GROUP BY a.Agent_ID, a.Name
ORDER BY Active_Listings DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.26 What percentage of buyers are investors vs end users?

In [ ]:
query = """
SELECT
    buyer_type,
    COUNT(*) AS Buyer_Count,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM buyers),
        2
    ) AS Percentage
FROM buyers
GROUP BY buyer_type
ORDER BY Percentage DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.27 Which cities have the highest loan uptake rate?

In [ ]:
query = """
SELECT
    l.City,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN b.loan_taken = 'Yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS Loan_Uptake_Rate
FROM buyers b
JOIN sales s
    ON b.sale_id = s.Listing_ID
JOIN listings l
    ON s.Listing_ID = l.Listing_id
GROUP BY l.City
ORDER BY Loan_Uptake_Rate DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.28 What is the average loan amount by buyer type?

In [ ]:
query = """
SELECT
    buyer_type,
    AVG(loan_amount) AS Average_Loan_Amount
FROM buyers
WHERE loan_taken = TRUE
GROUP BY buyer_type
ORDER BY Average_Loan_Amount DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.29 Which payment mode is most commonly used?

In [ ]:
query = """
SELECT
    payment_mode,
    COUNT(*) AS Usage_Count
FROM buyers
GROUP BY payment_mode
ORDER BY Usage_Count DESC;
"""

result = pd.read_sql(query, conn)

print(result)

Ques.30 Do loan-backed purchases take longer to close?

In [ ]:
query = """
SELECT
    payment_mode,
    COUNT(*) AS Usage_Count
FROM buyers
GROUP BY payment_mode
ORDER BY Usage_Count DESC;
"""

result = pd.read_sql(query, conn)

print(result)